**Universidad Central de Venezuela** Facultad de Ciencias Económicas y Sociales  
Escuela de Estadística y Ciencias Actuariales  
Asignatura: Computación I  

---

# Análisis de Juegos de Steam: ¿Cuáles son los mejores juegos?
**Tipo de trabajo:** Informe Académico  

**Autores:** Christopher Escalante, Gabriel Guevara, Noryen Harringhton  
**Profesores:** Jesús Ochoa, Oliver Triveño  
**Fecha:** Caracas, Abril 2026  

---

## Importación de librerias

In [147]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

## Cargas de datos

In [148]:
steam_data = pd.read_csv("Steam.csv")

## Limpieza de los Datos

In [149]:
steam_data['owners_min'] = steam_data['owners'].str.split('-').str[0].astype(float)
steam_data['owners_max'] = steam_data['owners'].str.split('-').str[1].astype(float)
steam_data['owners_avg'] = (steam_data['owners_min'] + steam_data['owners_max']) / 2

steam_data['ratio_positivo'] = steam_data['positive_ratings'] / (steam_data['positive_ratings'] + steam_data['negative_ratings'])

steam_data['modelo_negocio'] = np.where(steam_data['price'] == 0, "Free to Play", "Premium")

condiciones_tipo = [
    steam_data['steamspy_tags'].str.contains("Indie", na=False),
    steam_data['developer'].isin(["Valve", "Ubisoft", "Electronic Arts", "Activision", 
                                  "Bethesda", "Rockstar Games", "Capcom", "Square Enix", 
                                  "SEGA", "2K", "Bandai Namco"])
]
elecciones_tipo = ["Indie", "AAA"]
steam_data['tipo_juego'] = np.select(condiciones_tipo, elecciones_tipo, default="AA")

steam_data['multiplataforma'] = np.where(steam_data['platforms'].str.contains(";", na=False), "Multiplataforma", "Solo Windows")

condiciones_grupo = [
    steam_data['owners_avg'] < 1e6,
    steam_data['owners_avg'] < 5e6,
    steam_data['owners_avg'] < 1e7,
    steam_data['owners_avg'] < 2e7
]
elecciones_grupo = ["Menos de 1M", "1M - 5M", "5M - 10M", "10M - 20M"]
steam_data['owners_group'] = np.select(condiciones_grupo, elecciones_grupo, default="20M+")

steam_data['genre_main'] = steam_data['genres'].str.split(';').str[0]

steam_data.to_csv("steam_limpio_corregido.csv", index=False)
steam_data['owners_avg'] = (steam_data['owners_min'] + steam_data['owners_max']) / 2

steam_data['ratio_positivo'] = steam_data['positive_ratings'] / (steam_data['positive_ratings'] + steam_data['negative_ratings'])

steam_data['modelo_negocio'] = np.where(steam_data['price'] == 0, "Free to Play", "Premium")

condiciones_tipo = [
    steam_data['steamspy_tags'].str.contains("Indie", na=False),
    steam_data['developer'].isin(["Valve", "Ubisoft", "Electronic Arts", "Activision", 
                                  "Bethesda", "Rockstar Games", "Capcom", "Square Enix", 
                                  "SEGA", "2K", "Bandai Namco"])
]
elecciones_tipo = ["Indie", "AAA"]
steam_data['tipo_juego'] = np.select(condiciones_tipo, elecciones_tipo, default="AA")

steam_data['multiplataforma'] = np.where(steam_data['platforms'].str.contains(";", na=False), "Multiplataforma", "Solo Windows")

condiciones_grupo = [
    steam_data['owners_avg'] < 1e6,
    steam_data['owners_avg'] < 5e6,
    steam_data['owners_avg'] < 1e7,
    steam_data['owners_avg'] < 2e7
]
elecciones_grupo = ["Menos de 1M", "1M - 5M", "5M - 10M", "10M - 20M"]
steam_data['owners_group'] = np.select(condiciones_grupo, elecciones_grupo, default="20M+")

steam_data['genre_main'] = steam_data['genres'].str.split(';').str[0]

steam_data.to_csv("steam_limpio_corregido.csv", index=False)

In [150]:
steam_data = pd.read_csv("steam_limpio_corregido.csv")

## 1. Indie vs AAA: Comparativa del Ratio de Reseñas Positivas

**Pregunta de investigación:** ¿Logran los juegos con la etiqueta "Indie" superar en ratio de reseñas positivas a los juegos de grandes publicadores (AAA) a pesar de tener menor volumen de propietarios?

In [151]:
df_filtrado_violin = steam_data[steam_data['tipo_juego'].isin(["Indie", "AAA"])]

fig = px.violin(
    df_filtrado_violin,
    x="tipo_juego",
    y="ratio_positivo",
    color="tipo_juego",
    box=True,
    color_discrete_map={"Indie": "#2ca02c", "AAA": "#d62728"}
)

fig.update_layout(
    title="¿Los Indie superan a los AAA en valoración?",
    xaxis_title="Tipo de juego",
    yaxis_title="Ratio de reseñas positivas",
    yaxis_tickformat=".0%"
)

fig.show()

**Interpretación de los datos:**
Los juegos Indie muestran una mayor variabilidad en su ratio de reseñas positivas, con valores que van desde muy bajos hasta muy altos. Los AAA son más consistentes, con distribuciones más estrechas y generalmente altas. No se observa que los Indie superen claramente a los AAA, pero sí que algunos Indie alcanzan niveles de valoración tan altos como los mejores AAA.

## 2. Retención Real: Free to Play vs Premium

**Pregunta de investigación:** ¿Cómo se compara la retención real (tiempo de juego) de los gigantes Free to Play frente a los títulos Premium más exitosos? ¿El modelo gratuito genera ecosistemas donde solo sobreviven los juegos como servicio?

In [152]:
df_filtrado_dispersion = steam_data[
    (steam_data['owners_min'] > 0) & 
    (steam_data['average_playtime'] < 50000) & 
    (steam_data['modelo_negocio'].notna())
].copy()

df_filtrado_dispersion['horas_juego'] = df_filtrado_dispersion['average_playtime'] / 60

fig = px.scatter(
    df_filtrado_dispersion,
    x="owners_min",
    y="horas_juego",
    color="modelo_negocio",
    log_x=True,
    trendline="lowess",
    opacity=0.4,
    color_discrete_map={"Free to Play": "#4CAF50", "Premium": "#E53935"}
)

fig.update_layout(
    template="plotly_dark",
    paper_bgcolor="#121212",
    plot_bgcolor="#1E1E1E",
    title={
        'text': "<b>Relación entre popularidad y retención</b>",
        'y': 0.95,
        'x': 0.5,
        'xanchor': 'center',
        'yanchor': 'top',
        'font': dict(size=20, color="#E0E0E0")
    },
    xaxis_title="<b>Propietarios estimados (escala log)</b>",
    yaxis_title="<b>Tiempo promedio de juego (horas)</b>",
    yaxis=dict(
        range=[0, 800],
        showgrid=True,
        gridcolor="#333333"
    ),
    xaxis=dict(
        showgrid=True,
        gridcolor="#333333"
    ),
    font=dict(
        family="Helvetica, Arial, sans-serif",
        size=12,
        color="#B0B0B0"
    ),
    legend_title_text="Modelo de negocio"
)

fig.update_traces(
    selector=dict(mode='markers'),
    marker=dict(size=4)
)

fig.show()



**Interpretación de los datos:**
Los juegos Free to Play (F2P) muestran una retención mucho más alta a medida que crecen en popularidad, mientras que los Premium también retienen más cuando son exitosos, pero con una pendiente más suave. Esto sugiere que los gigantes F2P sí generan ecosistemas de altísima retención, típicos de juegos como servicio, pero no son los únicos: los Premium más exitosos también logran retención elevada, aunque con un patrón distinto.

## 3. Impacto del Precio en la Valoración del Usuario

**Pregunta de investigación:** ¿Existe una correlación directa entre el precio de un juego y su valoración, o los usuarios son más críticos con los juegos más caros?

In [153]:
df_filtrado_box = steam_data[(steam_data['price'] > 0) & (steam_data['price'] <= 500)].copy()
bins = [0, 5, 10, 20, 40, 60, 100, 500]
etiquetas = ["0-5", "5-10", "10-20", "20-40", "40-60", "60-100", "100-500"]
df_filtrado_box['price_bin'] = pd.cut(df_filtrado_box['price'], bins=bins, labels=etiquetas)

fig = px.box(
    df_filtrado_box,
    x="price_bin",
    y="ratio_positivo",
    color="price_bin",
    color_discrete_sequence=["#B71C1C", "#E62424", "#E53935", "#FF5252", "#81C784", "#4CAF50", "#2E7D32"]
)

fig.update_layout(
    template="plotly_dark",
    paper_bgcolor="#121212",
    plot_bgcolor="#1E1E1E",
    title={
        'text': "<b>Valoración según rangos de precio</b>",
        'y': 0.95,
        'x': 0.5,
        'xanchor': 'center',
        'yanchor': 'top',
        'font': dict(size=20, color="#E0E0E0")
    },
    xaxis_title="<b>Rango de precio (USD)</b>",
    yaxis_title="<b>Ratio de reseñas positivas</b>",
    yaxis=dict(
        tickformat=".0%",
        showgrid=True,
        gridcolor="#333333",
        zeroline=False
    ),
    xaxis=dict(
        showgrid=False,
        zeroline=False
    ),
    font=dict(
        family="Helvetica, Arial, sans-serif",
        size=12,
        color="#B0B0B0"
    ),
    showlegend=False,
    margin=dict(l=60, r=40, t=80, b=60)
)

fig.show()

**Interpretación de los datos:**
El precio no determina la valoración. Los juegos baratos, medianos y caros tienen medianas muy parecidas. No se observa que los usuarios sean más críticos con los juegos caros, ni que los juegos de mayor precio reciban mejores reseñas.

## 4. Concentración de Propietarios y Tiempo de Juego

**Pregunta de investigación:** ¿Cuáles son los juegos con mayor número de propietarios estimados? ¿Se traduce un alto número de descargas en altos promedios de tiempo de juego?

In [ ]:
top_propietarios = steam_data.nlargest(10, 'owners_avg').sort_values(by='owners_avg', ascending=False)

fig = px.bar(
    top_propietarios,
    x="name",
    y="owners_avg",
    color="name",
    color_discrete_sequence=["#2E7D32", "#388E3C", "#4CAF50", "#66BB6A", "#81C784", "#E57373", "#EF5350", "#E53935", "#C62828", "#B71C1C"]
)

fig.update_layout(
    height=800,
    template="plotly_dark",
    paper_bgcolor="#121212",
    plot_bgcolor="#1E1E1E",
    title={
        'text': "<b>Top 10 juegos con mayor número de propietarios estimados</b>",
        'y': 0.95,
        'x': 0.5,
        'xanchor': 'center',
        'yanchor': 'top',
        'font': dict(size=20, color="#E0E0E0")
    },
    xaxis_title="<b>Juego</b>",
    yaxis_title="<b>Propietarios estimados (promedio)</b>",
    yaxis=dict(
        tickformat=",",
        showgrid=True,
        gridcolor="#333333"
    ),
    xaxis=dict(
        showgrid=False,
        tickangle=-45 
    ),
    font=dict(
        family="Helvetica, Arial, sans-serif",
        size=12,
        color="#B0B0B0"
    ),
    showlegend=False,
    margin=dict(t=80, b=120, l=40, r=40) 
)

fig.show()

medianas = steam_data.groupby('owners_group', as_index=False)['median_playtime'].median()
medianas.rename(columns={'median_playtime': 'mediana'}, inplace=True)

orden_grupos = ["Menos de 1M", "1M - 5M", "5M - 10M", "10M - 20M", "20M+"]

fig = px.bar(
    medianas,
    x="owners_group",
    y="mediana",
    color="owners_group",
    category_orders={"owners_group": orden_grupos},
    color_discrete_sequence=["#B71C1C", "#E53935", "#FF5252", "#81C784", "#2E7D32"]
)

fig.update_layout(
    template="plotly_dark",
    paper_bgcolor="#121212",
    plot_bgcolor="#1E1E1E",
    title={
        'text': "<b>Mediana del tiempo de juego por grupo de propietarios</b>",
        'y': 0.95,
        'x': 0.5,
        'xanchor': 'center',
        'yanchor': 'top',
        'font': dict(size=20, color="#E0E0E0")
    },
    xaxis_title="<b>Grupo de propietarios</b>",
    yaxis_title="<b>Tiempo mediano de juego (minutos)</b>",
    yaxis=dict(
        tickformat=",",
        showgrid=True,
        gridcolor="#333333"
    ),
    xaxis=dict(
        showgrid=False
    ),
    font=dict(
        family="Helvetica, Arial, sans-serif",
        size=12,
        color="#B0B0B0"
    ),
    showlegend=False,
    margin=dict(t=80, b=40, l=40, r=40)
)

fig.update_traces(
    texttemplate='%{y:,.0f}',
    textposition='outside'
)

fig.show()


**Interpretación de los datos:**
El gráfico muestra una alta concentración de popularidad: pocos títulos acumulan la mayoría de propietarios. Esto describe el tamaño del mercado, pero no permite inferir compromiso ni tiempo de juego. Es un gráfico contextual, no explicativo. 

La mediana del tiempo de juego es similar en casi todos los grupos (200–350 min). Esto indica que el comportamiento típico del jugador no cambia significativamente según la popularidad del juego. El grupo 20M+ presenta una mediana mayor, pero esto se debe a títulos excepcionales, no a una tendencia general.

## 5. El Impacto del Soporte Multiplataforma

**Pregunta de investigación:** Evaluar cómo el soporte a múltiples plataformas (Windows, Mac, Linux) y la disponibilidad de idiomas afectan el tamaño del mercado potencial y las valoraciones de los usuarios.

In [155]:
agrupacion = steam_data.groupby('multiplataforma').agg(
    owners_avg_mean=('owners_avg', 'mean'),
    ratio_pos_mean=('ratio_positivo', 'mean')
).reset_index()

val_ventas_multi = agrupacion.loc[agrupacion['multiplataforma'] == 'Multiplataforma', 'owners_avg_mean'].values[0]
val_ventas_win = agrupacion.loc[agrupacion['multiplataforma'] == 'Solo Windows', 'owners_avg_mean'].values[0]
incremento_ventas = (val_ventas_multi - val_ventas_win) / val_ventas_win

val_ratio_multi = agrupacion.loc[agrupacion['multiplataforma'] == 'Multiplataforma', 'ratio_pos_mean'].values[0]
val_ratio_win = agrupacion.loc[agrupacion['multiplataforma'] == 'Solo Windows', 'ratio_pos_mean'].values[0]
incremento_ratio = (val_ratio_multi - val_ratio_win) / val_ratio_win

colores = {'Multiplataforma': '#4CAF50', 'Solo Windows': '#E53935'}

fig = make_subplots(rows=1, cols=2, subplot_titles=("Ventas estimadas", "Críticas positivas"))

for plataforma in agrupacion['multiplataforma']:
    df_sub = agrupacion[agrupacion['multiplataforma'] == plataforma]
    
    fig.add_trace(
        go.Bar(
            x=df_sub['multiplataforma'],
            y=df_sub['owners_avg_mean'],
            text=df_sub['owners_avg_mean'],
            texttemplate='%{text:,.0f}',
            textposition='outside',
            marker_color=colores[plataforma],
            showlegend=False
        ),
        row=1, col=1
    )
    
    fig.add_trace(
        go.Bar(
            x=df_sub['multiplataforma'],
            y=df_sub['ratio_pos_mean'],
            text=df_sub['ratio_pos_mean'],
            texttemplate='%{text:.1%}',
            textposition='outside',
            marker_color=colores[plataforma],
            showlegend=False
        ),
        row=1, col=2
    )

fig.update_layout(
    template="plotly_dark",
    paper_bgcolor="#121212",
    plot_bgcolor="#1E1E1E",
    font=dict(family="Helvetica, Arial, sans-serif", size=12, color="#B0B0B0"),
    margin=dict(t=80, b=40, l=40, r=40)
)

fig.update_yaxes(title_text="Promedio de propietarios estimados", showgrid=True, gridcolor="#333333", row=1, col=1)
fig.update_yaxes(title_text="Ratio de reseñas positivas", tickformat=".0%", showgrid=True, gridcolor="#333333", row=1, col=2)
fig.update_xaxes(title_text="Soporte de plataformas", row=1, col=1)
fig.update_xaxes(title_text="Soporte de plataformas", row=1, col=2)

fig.show()

**Interpretación de los datos:**
Los resultados muestran que los juegos multiplataforma presentan un promedio de 210.840 propietarios, mientras que los juegos exclusivos de Windows alcanzan aproximadamente 97.918 propietarios. En cuanto a las críticas positivas, los juegos multiplataforma alcanzan un ratio promedio de 75,8%, frente al 69,4% de los juegos exclusivos de Windows.

## 6. Nichos de Oportunidad para Desarrolladores

**Pregunta de investigación:** Desde una óptica de desarrollo de producto, ¿qué combinación de Género y Categoría presenta la menor saturación en el mercado pero la mayor tasa de reseñas positivas?

In [156]:
combo = steam_data.groupby(['genre_main', 'tipo_juego']).agg(
    n_juegos=('tipo_juego', 'count'),
    ratio_prom=('ratio_positivo', 'mean')
).reset_index()

combo['indice_oportunidad'] = combo['ratio_prom'] / np.log(combo['n_juegos'] + 1)
combo['combo_label'] = combo['genre_main'] + " - " + combo['tipo_juego']

combo_top15 = combo.nlargest(15, 'indice_oportunidad').sort_values(by='indice_oportunidad', ascending=False)

fig = px.bar(
    combo_top15,
    x="combo_label",
    y="indice_oportunidad",
    color="tipo_juego",
    color_discrete_map={"Indie": "#4CAF50", "AAA": "#E53935", "AA": "#F1F106"}
)

fig.update_layout(
    height=750,
    template="plotly_dark",
    paper_bgcolor="#121212",
    plot_bgcolor="#1E1E1E",
    title={
        'text': "<b>Top 15 combinaciones por Índice de Oportunidad</b>",
        'y': 0.95,
        'x': 0.5,
        'xanchor': 'center',
        'yanchor': 'top',
        'font': dict(size=20, color="#E0E0E0")
    },
    xaxis_title="<b>Combinación</b>",
    yaxis_title="<b>Índice de oportunidad</b>",
    yaxis=dict(
        showgrid=True,
        gridcolor="#333333"
    ),
    xaxis=dict(
        showgrid=False,
        tickangle=-45
    ),
    font=dict(
        family="Helvetica, Arial, sans-serif",
        size=12,
        color="#B0B0B0"
    ),
    legend_title_text="Categoría",
    margin=dict(t=80, b=150, l=40, r=40)
)

fig.show()

**Interpretación de los datos:**
El gráfico muestra que en las primeras posiciones —como Software Training – AA, Animation & Modeling – AAA y Web Publishing – AA— destacan porque tienen pocos competidores y una recepción muy positiva, por lo que representan los nichos con mayor potencial para nuevos desarrollos.